# A Abordagem de Feature Store

Para organizar as variáveis utilizadas no modelo, foi adotada uma abordagem baseada em **Feature Store**. O objetivo é separar a **construção das features** da etapa de modelagem, permitindo que as mesmas variáveis sejam utilizadas de forma consistente tanto no treinamento quanto na previsão.

Construção das features:

A criação das variáveis é realizada utilizando **queries SQL**, armazenadas em arquivos *.sql* separados. Essa organização permite manter a lógica de cada grupo de features isolada e facilita sua manutenção e reutilização.

A estrutura segue, por exemplo:

```text
feature_store/
├── fs_cadastral.sql
├── fs_temporal.sql
├── fs_historico_financeiro.sql
├── fs_renda.sql
├── fs_funcionarios.sql
└── fs_historico_pagamentos.sql
```

As queries são parametrizadas pelo período de referência. Dessa forma, a mesma consulta pode ser executada para diferentes meses, gerando as features correspondentes a cada DATA_REF.

Ingestão no Feature Store:

A execução das queries é centralizada em um **notebook de ingestão**. Esse notebook lê cada arquivo *.sql*, executa a consulta para os períodos definidos e grava os resultados nas respectivas tabelas do Feature Store.

O fluxo pode ser representado da seguinte forma:

```text
Dados brutos
     ↓
Queries SQL
     ↓
Construção das features
     ↓
Notebook de ingestão
     ↓
Feature Store
     ↓
Training Set / Predição
```

Na primeira execução, caso a tabela ainda não exista, ela é criada definindo:

* *ID_CLIENTE*
* *ID_DOCUMENTO*
* *DATA_REF*

como chaves da feature;

* *DATA_REF* como coluna de particionamento.

Nas execuções seguintes, os novos períodos são adicionados utilizando **merge**, permitindo atualizar o Feature Store sem precisar recriar toda a tabela.

Utilização no treinamento:

Durante o treinamento, as diferentes tabelas de features são recuperadas através do *FeatureEngineeringClient* e relacionadas à base principal utilizando as chaves definidas.

Isso permite construir um **training set único**, combinando as informações cadastrais, temporais e históricas necessárias para o modelo.

Uma vantagem importante dessa abordagem é garantir que a construção das features seja **reprodutível e organizada**, além de facilitar a utilização das mesmas features posteriormente no processo de previsão.

> **Importante:** como se trata de um problema temporal, as features históricas devem ser construídas utilizando apenas informações disponíveis até a respectiva *DATA_REF*. Isso evita que informações futuras sejam utilizadas durante o treinamento e reduz o risco de *data leakage*.



As features foram divididas em diferentes grupos de acordo com sua origem e finalidade:

* **Cadastral**: características cadastrais dos clientes;
* **Temporal**: informações relacionadas ao período de referência;
* **Histórico financeiro**: características relacionadas ao comportamento financeiro;
* **Renda**: informações e variações relacionadas à renda;
* **Funcionários**: histórico e comportamento do número de funcionários;
* **Histórico de pagamentos**: métricas relacionadas ao comportamento de pagamento.


# Ideias de Features

## Feature Store Cadastral


**Chave:** ID_CLIENTE

Features relacionadas às características cadastrais e ao relacionamento do cliente com a empresa.

**Features**

* **Região do cliente:** região geográfica obtida a partir das informações cadastrais.
* **Tempo de relacionamento:** tempo decorrido entre a data de cadastro do cliente e a data de referência.


## Feature Store Renda


**Chave:** ID_CLIENTE, SAFRA_REF

Features destinadas a representar o **nível, comportamento, tendência e estabilidade da renda do cliente ao longo do tempo**.

**Agregações de Renda**

São calculadas estatísticas da renda considerando diferentes janelas temporais.

3 meses

* Média da renda.
* Soma da renda.
* Menor renda.
* Maior renda.

6 meses

* Média da renda.
* Soma da renda.
* Menor renda.
* Maior renda.

12 meses

* Média da renda.
* Soma da renda.
* Menor renda.
* Maior renda.

Vida toda

* Média histórica da renda.
* Soma histórica da renda.
* Menor renda histórica.
* Maior renda histórica.

**Tendências**

Buscam identificar como a renda do cliente está evoluindo ao longo do tempo, comparando os períodos anteriores com períodos mais recentes.

* Crescimento da renda em 3 meses.
* Crescimento da renda em 6 meses.
* Crescimento da renda em 12 meses.
* Redução da renda em 3 meses.
* Redução da renda em 6 meses.
* Redução da renda em 12 meses.

Essas medidas são calculadas de forma **temporal**, utilizando somente os períodos anteriores à SAFRA_REF.

**Variabilidade**

Buscam medir a **estabilidade da renda** do cliente.

* Desvio padrão da renda em 3 meses.
* Desvio padrão da renda em 6 meses.
* Desvio padrão da renda em 12 meses.
* Coeficiente de variação da renda.

**Histórico**

Busca identificar períodos prolongados de redução da renda, representando possíveis mudanças no comportamento financeiro do cliente.

* **Meses consecutivos de queda:** quantidade de meses consecutivos em que a renda apresentou redução em relação ao período anterior.

> As janelas e métricas apresentadas representam hipóteses iniciais. Durante o desenvolvimento, elas poderão ser ajustadas, removidas ou complementadas de acordo com a disponibilidade dos dados e os resultados da análise.


## Feature Store Funcionários


**Chave:** ID_CLIENTE, SAFRA_REF

A Feature Store de funcionários tem como objetivo representar o **tamanho, evolução e estabilidade do quadro de funcionários** do cliente ao longo do tempo. Essas informações podem ajudar o modelo a identificar mudanças na estrutura da empresa que estejam relacionadas ao risco de inadimplência.

**Features**

Quadro atual

* Número atual de funcionários.

Crescimento do quadro

Calcula a evolução do número de funcionários em diferentes janelas temporais:

* Crescimento do quadro nos últimos 3 meses.
* Crescimento do quadro nos últimos 6 meses.
* Crescimento do quadro nos últimos 12 meses.
* Crescimento do quadro ao longo de todo o histórico disponível.

Redução do quadro

Identifica reduções no número de funcionários:

* Redução do quadro nos últimos 3 meses.
* Redução do quadro nos últimos 6 meses.
* Redução do quadro nos últimos 12 meses.
* Redução do quadro ao longo de todo o histórico disponível.

Estatísticas históricas

Busca identificar variações relevantes no quadro de funcionários:

* Maior crescimento mensal observado.
* Maior queda mensal observada.

Comparação com o porte

Compara o número atual de funcionários do cliente com o comportamento esperado para empresas do mesmo porte:

* Diferença entre o número atual de funcionários e a média de funcionários do respectivo porte.

Eficiência

Relaciona o tamanho do quadro com a renda da empresa:

* **Renda por funcionário:** renda mensal dividida pelo número de funcionários.

> **Observação:** essas são as features inicialmente propostas. Durante a exploração e modelagem, novas variáveis podem ser criadas, algumas podem ser modificadas e outras podem ser descartadas caso apresentem pouca relevância, problemas de disponibilidade ou risco de *data leakage*.


## Feature Store Historico de Pagamentos


**Chave:** ID_CLIENTE, SAFRA_REF

A Feature Store de histórico de pagamentos tem como objetivo representar o **comportamento financeiro e a pontualidade do cliente ao longo do tempo**. As variáveis são construídas a partir do histórico de cobranças e pagamentos, utilizando diferentes janelas temporais.

**Features**

Atrasos

Busca identificar a frequência e a intensidade dos atrasos do cliente:

* Flag de atraso nos últimos 3 meses.
* Flag de atraso nos últimos 6 meses.
* Flag de atraso nos últimos 12 meses.
* Flag de atraso ao longo de todo o histórico.
* Quantidade de atrasos nos últimos 3 meses.
* Quantidade de atrasos nos últimos 6 meses.
* Quantidade de atrasos nos últimos 12 meses.
* Quantidade de atrasos ao longo de todo o histórico.
* Dias desde o último atraso.
* Média de dias de atraso.
* Maior quantidade de dias de atraso.

Pagamentos

Caracteriza o comportamento de pagamento do cliente:

* Dias desde o último pagamento.
* Quantidade de pagamentos antecipados.
* Média de dias de antecipação.
* Quantidade de pagamentos realizados no vencimento.

Boletos

Representa o volume e os valores das cobranças:

* Quantidade de boletos.
* Valor médio dos boletos.
* Maior valor de boleto.
* Valor total pago nos últimos 3 meses.
* Valor total pago nos últimos 6 meses.
* Valor total pago nos últimos 12 meses.

Pontualidade

Mede o comportamento do cliente em relação aos prazos de pagamento:

* Percentual de pagamentos realizados em dia.

Datas

Caracteriza os intervalos relacionados ao ciclo de cobrança e pagamento:

* Prazo médio entre emissão e vencimento.
* Prazo mínimo entre emissão e vencimento.
* Prazo máximo entre emissão e vencimento.
* Dias restantes até o vencimento.
* Média de dias entre emissão e pagamento.

Cobrança

Relaciona o valor da cobrança com o prazo disponível para pagamento:

* Valor da cobrança por dia.

> **Observação:** essas são as features inicialmente propostas. A definição final poderá sofrer alterações durante a exploração e modelagem, incluindo criação de novas variáveis, remoção de variáveis pouco relevantes e ajustes nas janelas temporais. As features também devem respeitar a disponibilidade das informações no momento da previsão, evitando *data leakage*.


## Feature Store Temporal


**Chave:** ID_CLIENTE, SAFRA_REF

A Feature Store de features temporais tem como objetivo representar a **relação do cliente com o tempo**, utilizando informações disponíveis no momento da previsão. Essas variáveis ajudam o modelo a capturar aspectos relacionados ao tempo de relacionamento e ao ciclo de cobrança.

Features

Relacionamento

* Dias desde o cadastro do cliente.

Cobrança

* Dias até o vencimento.
* Prazo entre emissão e vencimento.

Histórico de pagamentos

* Dias desde o último pagamento.
* Dias desde o último atraso.

> **Observação:** essas são as features inicialmente propostas. A definição final poderá ser alterada durante a exploração e modelagem, conforme a disponibilidade das informações, relevância preditiva e necessidade de evitar *data leakage*.
